# Datamine MDTRAN Process Example

This notebook demonstrates how to configure and run the **`mdtran`** process wrapper in `dmstudio`.

## Process Description

## MDTRAN

MDTRAN creates a new block model on a different prototype and coordinate system to an existing block model.

The coordinate system for the new model may be translated and rotated with respect to the existing system.

The parameters to define the translation and rotation are the same as those used in the [CDTRAN](<cdtran.md>) process. Three rotations about different axes may be specified.

There are three MDTRAN scenarios:

|  &IN |  &PROTO |  &PROTOROT or Parameters |  &OUT
---|---|---|---|---
1 | Local grid, but not a rotated model | World grid | Rotations INVERSE=0 | World grid
2 | Rotated model | World grid | &IN rotated model prototype INVERSE=1 | World grid
3 | World grid | Rotated model prototype | &PROTO rotated model prototype INVERSE=-0 | Rotated model

* To convert from a local grid (without rotated model fields) to a rotated model simply append the rotated model prototype to the local grid model.
* It is not possible to directly convert from a rotated model to another rotated model with different rotations. To do this you would have to run **MDTRAN** twice, first for scenario 2 then for scenario 3.
* Usually, &**PROTO** will not contain records. If it does then only cells at &**PROTO** locations will be assigned values from &**IN**.
* The rotations can be defined using either &**PROTOROT** or the MDTRAN parameters. If the &**PROTOROT** file is defined then its values take precedence over the parameters.
* It would seem that **MDTRAN** is set up for scenario 1 rather than 2\. In scenario 1 there are no rotated model fields in &**IN** so they are not copied to &OUT.
* In scenario 2 the rotated model fields in &**IN** are not identified as rotated model fields so they get copied from &**IN** to &**OUT** ; then they have to be removed.

The centre point of each subcell in the output model is identified, and the subcell values are set equal to the corresponding point in the input model. If the &**PROTO** model file contains records then the &**OUT** model will have the same configuration of cells/subcells, but with values taken from the &IN model. If there is no corresponding point in the &IN model, then absent data values will be assigned in the &**OUT** model.

If &**PROTO** is empty then the process will create subcells. Initially the entire model framework will be filled with cells and subcells as defined by the @**X/Y/ZSUBCELL** parameters. However only those subcells whose midpoints lie within a subcell of the &IN model will be assigned values and written to the &**OUT** model.

The assignment of values from the &**IN** model to the &**OUT** model is illustrated in Figure 2 (Examples section).

The translation and rotation are usually defined by the 12 parameters @**X0** , @**Y0** , @**Z0** , @**XR0** , @**YR0** , @**ZR0** , @**ANGLE1** , @**ANGLE2** , @**ANGLE3** , @**ROTAXIS1** , @**ROTAXIS2** , and @**ROTAXIS3**. Alternatively, a rotated model file may be defined as the optional &**PROTOROT** file. The data definition of this file will contain all the necessary translation and rotation information.

* **Note** (A macro illustrating this is included in the _Examples_ section below.):

A scaling function is provided so that for example meters may be converted to feet, miles to kilometers, etc. If the optional @**FACTOR** parameter is set, then the units of the points in the rotated system will be @**FACTOR** times the units of the unrotated system. This means that, for example, if @**FACTOR** = 0.3048 and input units are feet then the output units will be meters. @FACTOR is the number of output units which equals one unit. The input point X0, Y0, Z0 will be in input units, and XR0, YR0, ZR0 will be in output units.

The @INVERSE parameter allows an inverse coordinate rotation. If @**INVERSE** =1 then the input model &**IN** is assumed to be in the rotated system, and the output model &**OUT** will be in the unrotated system. Otherwise the parameters @**X0** , @**Y0** , @**Z0** , @**XR0** , @**YR0** , @**ZR0** , @**ANGLE1** , @**ANGLE2** , @**ANGLE3** , @**ROTAXIS1** , @**ROTAXIS2** , and @**ROTAXIS3** are set exactly as they would be for the original conversion from the unrotated to the rotated system. In other words, X0, Y0 and Z0 is still the point in the unrotated system which matches point XR0, YR0, ZR0 in the rotated system, and ANGLE1, ANGLE2 and ANGLE3 are the angles to rotate from the unrotated system to the rotated.

### Input Files:

* **in** (Block Model):
  Input model to be rotated. Must contain at least the fields XC, YC, ZC, XINC, YINC,
  ZINC, XMORIG, YMORIG, ZMORIG, NX, NY, NZ, and IJK. May also contain value fields. It
  must be sorted by IJK.
  Required=Yes

* **proto** (Model Prototype File):
  Prototype model defining output model. Must contain at least the fields **XC, YC, ZC,
  XINC, YINC, ZINC, XMORIG, YMORIG, ZMORIG, NX, NY, NZ** and **IJK**. May contain cells
  and subcells. Any fields which are in **PROTO** but not in **IN** will have their values
  carried across into **OUT**.
  Required=Yes

* **protorot** (Model Prototype File):
  Optional file containing the rotation and translation parameters stored as the default
  of implicit fields **ANGLE1, ANGLE2, ANGLE3, X0, Y0, Z0, XMORIG, YMORIG, ZMORIG,
  ROTAXIS1, ROTAXIS2** and **ROTAXIS3**. Fields **XMORIG, YMORIG** and **ZMORIG**
  correspond to parameters **XR0, YR0** and **ZR0**. The other nine fields have the same
  name as the corresponding parameters. If this file is specified and has valid values for
  all twelve fields then the parameter entries for rotation and translation are ignored.
  This file can be created using the Rotated Model option in process
  **[PROTOM](<protom.md>)**. Data will then be transformed into the local (rotated)
  coordinate system of the model.
  Required=No

### Output Files:

* **out** (Block Model):
  Output model. Will have default field values from PROTO for XC, YC, ZC, XINC, YINC,
  ZINC, XMORIG, YMORIG, ZMORIG, NX, NY, and NZ. Will also contain any value fields from IN
  and PROTO. It will be sorted by IJK.
  Required=Yes

### Fields:

### Parameters:

* **angle1**:
  First rotation angle clockwise in degrees, around axis **ROTAXIS1**. It must lie
  between -360.0 and +360.0. A value of zero indicates no rotation. (0)
  Range=-360, 360
  Values=Undefined
  Default=0
  Required=No

* **angle2**:
  Second rotation angle clockwise in degrees, around axis **ROTAXIS2**. It must lie
  between 360.0 and +360.0. A value of zero indicates no rotation. (0)
  Range=-360, 360
  Values=Undefined
  Default=0
  Required=No

* **angle3**:
  Third rotation angle clockwise in degrees, around axis **ROTAXIS3**. It must lie
  between -360.0 and +360.0. A value of zero indicates no rotation. (0)
  Range=-360, 360
  Values=Undefined
  Default=0
  Required=No

* **rotaxis1**:
  Axis around which first rotation angle will occur. 0 for no rotation, 1 for X axis, 2
  for Y axis, 3 for Z axis. (3)
  Range=0,3
  Values=0,1,2,3
  Default=3
  Required=No

* **rotaxis2**:
  Axis around which second rotation angle will occur. 0 for no rotation, 1 for X axis, 2
  for Y axis, 3 for Z axis. (1)
  Range=0,3
  Values=0,1,2,3
  Default=1
  Required=No

* **rotaxis3**:
  Axis around which third rotation angle will occur. 0 for no rotation, 1 for X axis, 2
  for Y axis, 3 for Z axis. (3)
  Range=0,3
  Values=0,1,2,3
  Default=3
  Required=No

* **x0**:
  X co-ordinate of known point in both systems, in unrotated co-ordinate system (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **y0**:
  Y co-ordinate of known point in both systems, in unrotated co-ordinate system (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **z0**:
  Z co-ordinate of known point in both systems, in unrotated co-ordinate system (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **xr0**:
  X co-ordinate of known point in both systems, in rotated co-ordinate system (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **yr0**:
  Y co-ordinate of known point in both systems, in rotated co-ordinate system (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **zr0**:
  Z co-ordinate of known point in both systems, in rotated co-ordinate system (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **xsubcell**:
  Cell division in X direction in OUT. Only used if **PROTO** is empty. Default (1), max
  20.
  Range=1,20
  Values=Undefined
  Default=1
  Required=No

* **ysubcell**:
  Cell division in Y direction in OUT. Only used if **PROTO** is empty. Default (1), max
  20.
  Range=1,20
  Values=Undefined
  Default=1
  Required=No

* **zsubcell**:
  Cell division in Z direction in OUT. Only used if **PROTO** is empty. Default (1), max
  20.
  Range=1,20
  Values=Undefined
  Default=1
  Required=No

* **factor**:
  Co-ordinate scaling factor. Default (1). The rotated co-ordinate system units will be
  e.g. 0.3048 for a grid in metres on an unrotated grid in feet.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **inverse**:
  Inverse transformation. Default (0). Options: 0: Rotate from IN through {ANGLE1,
  ANGLE2,ANGLE3} to OUT.; 1: Inverse transformation to above; OUT is in rotated system; IN
  is in unrotated system; ANGLE1-3 are same angles as for 0.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **print**:
  Print flag. Default (0). 0 - minimum output. 1 - details of each subcell in output
  model.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('mdtran')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute mdtran
print("Running mdtran...")
dm_cmd.mdtran(
    in_i='t_assays',  # required
    proto_i='t_mod1',  # required
    out_o='t_mdtran_out',  # required
    # protorot_i='t_mod1',  # optional
    # angles_f=['optional'],  # optional
    # rotaxiss_f=['optional'],  # optional
    # x0_p=0,  # optional
    # y0_p=0,  # optional
    # z0_p=0,  # optional
    # xr0_p=0,  # optional
    # yr0_p=0,  # optional
    # zr0_p=0,  # optional
    # xsubcell_p=1,  # optional
    # ysubcell_p=1,  # optional
    # zsubcell_p=1,  # optional
    # factor_p=1,  # optional
    # inverse_p=0,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("mdtran execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_mdtran_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")